### NumPy Exercises: Supermarket Sales

#### 📋 Dataset Information

**Dataset**: Supermarket Sales  
**Source**: Attached file  

**Columns**:
- `Branch`: Store branch identifier where the transaction occurred ('A', 'B', or 'C').
- `Customer type`: Customer membership status ('Member', 'Normal').
- `Gende`r: Customer's gender ('Male' or 'Female').
- `Product line`: Product category purchased (6 categories: Electronic accessories, Fashion accessories, Food and beverages, Health and beauty, Home and lifestyle, Sports and travel).
- `Quantity`: Number of items purchased in the transaction.
- `Total`: Total amount paid by customer in dollars, including tax.
- `Date`: Transaction date in M/D/YYYY format.
- `Rating`: Customer satisfaction rating on a scale of 0.0 to 10.0 (higher is better).

In [1]:
import numpy as np

In [2]:
## NO NEED CODE HERE
# Load the CSV file
data = np.genfromtxt(
    'Data/supermarket_sales.csv',
    delimiter=',',
    skip_header=1,    # Skip header row
    dtype=str         # Load as strings (no preprocessing)
)

# Print basic info
print("="*60)
print("DATASET LOADED")
print("="*60)
print(f"Shape: {data.shape}")
print(f"  Rows: {data.shape[0]} transactions")
print(f"  Columns: {data.shape[1]}")

# Column names
columns = ['Branch', 'Customer type', 'Gender', 'Product line', 
           'Quantity', 'Total', 'Date', 'Rating']

print(f"\nColumns:")
for i, col in enumerate(columns):
    print(f"  [{i}] {col}")

DATASET LOADED
Shape: (1000, 8)
  Rows: 1000 transactions
  Columns: 8

Columns:
  [0] Branch
  [1] Customer type
  [2] Gender
  [3] Product line
  [4] Quantity
  [5] Total
  [6] Date
  [7] Rating


### Task 1: Customer Segmentation Analysis
**Which customer segment (by type and gender) generates the highest revenue per transaction, and how does their rating behavior differ from other segments?** 

**Instruction:**
- 4 customer segments: Member-Female, Member-Male, Normal-Female, Normal-Male.
- Calculate revenue per transaction for each segment.
- Calculate average rating for each segment.
- How each segment compares to the overall average?
   - Calculate deviations 
- Which segment is the "best"? (Based on revenue, rating and count)
   - Weight: 50% revenue, 30% rating, 20% count (volume) (Need normalize to [0.1])

In [ ]:
#TODO: your code here

customer_type = data[:, 1]  # Customer type
gender = data[:, 2]         # Gender
total = data[:, 5].astype(float)  # Total
rating = data[:, 7].astype(float) # Rating

# Define 4 segments
segments = {
    'Member-Female': (customer_type == 'Member') & (gender == 'Female'),
    'Member-Male': (customer_type == 'Member') & (gender == 'Male'),
    'Normal-Female': (customer_type == 'Normal') & (gender == 'Female'),
    'Normal-Male': (customer_type == 'Normal') & (gender == 'Male')
}

# Calculate metrics for each segment
segment_results = {}
for seg_name, mask in segments.items():
    count = np.sum(mask)
    avg_revenue = np.mean(total[mask])
    avg_rating = np.mean(rating[mask])
    
    segment_results[seg_name] = {
        'count': count,
        'avg_revenue': avg_revenue,
        'avg_rating': avg_rating
    }

# Overall averages
overall_avg_revenue = np.mean(total)
overall_avg_rating = np.mean(rating)

print(f"\nOverall Averages:")
print(f"  Revenue per transaction: ${overall_avg_revenue:.2f}")
print(f"  Rating: {overall_avg_rating:.2f}")
print(f"\nSegment Performance:")

for seg_name, metrics in segment_results.items():
    revenue_dev = ((metrics['avg_revenue'] - overall_avg_revenue) / overall_avg_revenue) * 100
    rating_dev = ((metrics['avg_rating'] - overall_avg_rating) / overall_avg_rating) * 100
    
    print(f"\n{seg_name}:")
    print(f"  Count: {metrics['count']} transactions ({metrics['count']/len(data)*100:.1f}%)")
    print(f"  Avg Revenue: ${metrics['avg_revenue']:.2f} ({revenue_dev:+.1f}% vs overall)")
    print(f"  Avg Rating: {metrics['avg_rating']:.2f} ({rating_dev:+.1f}% vs overall)")

# Calculate composite score (50% revenue, 30% rating, 20% count)
print(f"\n{'Segment Ranking (Weighted Score)':^60}")
print("-"*60)

# Normalize metrics to [0, 1]
revenues = np.array([m['avg_revenue'] for m in segment_results.values()])
ratings = np.array([m['avg_rating'] for m in segment_results.values()])
counts = np.array([m['count'] for m in segment_results.values()])

norm_revenues = (revenues - revenues.min()) / (revenues.max() - revenues.min())
norm_ratings = (ratings - ratings.min()) / (ratings.max() - ratings.min())
norm_counts = (counts - counts.min()) / (counts.max() - counts.min())

scores = {}
for i, seg_name in enumerate(segment_results.keys()):
    score = 0.5 * norm_revenues[i] + 0.3 * norm_ratings[i] + 0.2 * norm_counts[i]
    scores[seg_name] = score

# Sort by score
sorted_segments = sorted(scores.items(), key=lambda x: x[1], reverse=True)

for rank, (seg_name, score) in enumerate(sorted_segments, 1):
    print(f"{rank}. {seg_name:20s} - Score: {score:.3f}")

print(f"\n🏆 BEST SEGMENT: {sorted_segments[0][0]}")

### Task 2: Product Performance Optimization
**Which product lines are underperforming (below average sales) in which branches, and what is the revenue opportunity if they reached branch-average performance?**

**Instructions**:
1. **18 combinations**: 3 Branches × 6 Product Lines = 18
2. **Average sales per product-branch combination**
3. **Identify underperformers**: Which are below their branch average?
4. **Calculate gap**: How much below average?
5. **Revenue opportunity**: Potential gain if they reached branch average

In [ ]:
#TODO: your code here

branch = data[:, 0]
product_line = data[:, 3]

# Get unique branches and products
unique_branches = np.unique(branch)
unique_products = np.unique(product_line)

print(f"\nBranches: {', '.join(unique_branches)}")
print(f"Product Lines: {len(unique_products)} categories")

# Calculate average sales for each branch-product combination
branch_product_sales = {}
branch_averages = {}

for b in unique_branches:
    branch_mask = branch == b
    branch_total = total[branch_mask]
    branch_avg = np.mean(branch_total)
    branch_averages[b] = branch_avg
    
    for p in unique_products:
        combo_mask = (branch == b) & (product_line == p)
        if np.sum(combo_mask) > 0:
            avg_sales = np.mean(total[combo_mask])
            count = np.sum(combo_mask)
            branch_product_sales[(b, p)] = {
                'avg_sales': avg_sales,
                'count': count,
                'branch_avg': branch_avg
            }

# Identify underperformers
print(f"\n{'UNDERPERFORMING PRODUCTS BY BRANCH':^60}")
print("-"*60)

total_opportunity = 0
for b in unique_branches:
    print(f"\nBranch {b} (Average: ${branch_averages[b]:.2f}):")
    branch_opportunity = 0
    
    for p in unique_products:
        key = (b, p)
        if key in branch_product_sales:
            data_point = branch_product_sales[key]
            if data_point['avg_sales'] < data_point['branch_avg']:
                gap = data_point['branch_avg'] - data_point['avg_sales']
                opportunity = gap * data_point['count']
                branch_opportunity += opportunity
                
                print(f"  ❌ {p:25s}: ${data_point['avg_sales']:7.2f} "
                      f"(${gap:6.2f} below avg, ${opportunity:8.2f} opportunity)")
    
    total_opportunity += branch_opportunity
    print(f"  Branch {b} Total Opportunity: ${branch_opportunity:.2f}")

print(f"\n💰 TOTAL REVENUE OPPORTUNITY: ${total_opportunity:.2f}")

### Task 3: High-Value Customer Identification
**What percentage of total revenue comes from the top 20% of transactions, and what are the common characteristics of these high-value transactions?**

**Instructions**:\
**1. Identify the Top 20% of Transactions by Total amount**

**2. Calculate Revenue Concentration**\
Determine what percentage of total revenue is generated by these top 20% of transactions.

**3. Profile High-Value Transaction Characteristics**\
For the top 20% transactions, analyze their common patterns across multiple dimensions. How many transaction happen: 
 - On each branches
 - On each product lines
 - On each customer types
 - On each gender

**4. Compare High-Value vs Overall Distribution**\
Calculate how the characteristics of high-value transactions differ from the overall dataset by comparing percentage distributions across branches, products, customer types, and gender. \
For example, if Branch A represents 40% of high-value transactions but only 33% of all transactions, this indicates Branch A has a premium customer base. 

**5. Calculate Average Metrics for High-Value Segment**\
Compute the average transaction amount, average quantity purchased, and average rating specifically for the top 20% group and compare these averages to the overall dataset averages. For example:
- Top 20% spend 125% MORE per transaction
- Top 20% buy 55% MORE items per transaction
- Top 20% Ratings ....
  
**6. Identify the "Golden Combination"**\
Find the most common combination of characteristics (Branch + Product + Customer Type + Gender) within the high-value segment

**7. Analyze Contribution by Percentile Groups**\
Beyond just the top 20%, how revenue is distributed across all percentile groups (top 20%, 20-40%, 40-60%, 60-80%, bottom 20%) 

In [ ]:
#TODO: your code here

sorted_indices = np.argsort(total)[::-1]
top_20_count = int(len(total) * 0.2)
top_20_indices = sorted_indices[:top_20_count]
top_20_mask = np.zeros(len(total), dtype=bool)
top_20_mask[top_20_indices] = True

# 2. Revenue concentration
top_20_revenue = np.sum(total[top_20_mask])
total_revenue = np.sum(total)
revenue_percentage = (top_20_revenue / total_revenue) * 100

print(f"\n📊 Revenue Concentration:")
print(f"  Top 20% transactions: {top_20_count} out of {len(total)}")
print(f"  Revenue from top 20%: ${top_20_revenue:.2f} ({revenue_percentage:.1f}% of total)")

# 3. Profile high-value transactions
print(f"\n🎯 High-Value Transaction Profile:")

quantity = data[:, 4].astype(int)

# Branch distribution
print("\n  Branch Distribution:")
for b in unique_branches:
    high_val_count = np.sum((branch == b) & top_20_mask)
    overall_count = np.sum(branch == b)
    high_val_pct = (high_val_count / top_20_count) * 100
    overall_pct = (overall_count / len(data)) * 100
    diff = high_val_pct - overall_pct
    print(f"    {b}: {high_val_pct:.1f}% (Overall: {overall_pct:.1f}%, {diff:+.1f}% diff)")

# Product line distribution
print("\n  Product Line Distribution:")
for p in unique_products:
    high_val_count = np.sum((product_line == p) & top_20_mask)
    overall_count = np.sum(product_line == p)
    high_val_pct = (high_val_count / top_20_count) * 100
    overall_pct = (overall_count / len(data)) * 100
    diff = high_val_pct - overall_pct
    print(f"    {p:25s}: {high_val_pct:.1f}% (Overall: {overall_pct:.1f}%, {diff:+.1f}%)")

# Customer type & Gender
print("\n  Customer Type:")
for ct in ['Member', 'Normal']:
    high_val_count = np.sum((customer_type == ct) & top_20_mask)
    high_val_pct = (high_val_count / top_20_count) * 100
    overall_pct = (np.sum(customer_type == ct) / len(data)) * 100
    print(f"    {ct}: {high_val_pct:.1f}% (Overall: {overall_pct:.1f}%)")

print("\n  Gender:")
for g in ['Male', 'Female']:
    high_val_count = np.sum((gender == g) & top_20_mask)
    high_val_pct = (high_val_count / top_20_count) * 100
    overall_pct = (np.sum(gender == g) / len(data)) * 100
    print(f"    {g}: {high_val_pct:.1f}% (Overall: {overall_pct:.1f}%)")

# 5. Average metrics comparison
print(f"\n📈 Average Metrics Comparison:")
top_20_avg_total = np.mean(total[top_20_mask])
top_20_avg_qty = np.mean(quantity[top_20_mask])
top_20_avg_rating = np.mean(rating[top_20_mask])

overall_avg_qty = np.mean(quantity)

total_diff = ((top_20_avg_total - overall_avg_revenue) / overall_avg_revenue) * 100
qty_diff = ((top_20_avg_qty - overall_avg_qty) / overall_avg_qty) * 100
rating_diff = ((top_20_avg_rating - overall_avg_rating) / overall_avg_rating) * 100

print(f"  Transaction Amount: ${top_20_avg_total:.2f} vs ${overall_avg_revenue:.2f} overall ({total_diff:+.1f}% MORE)")
print(f"  Quantity: {top_20_avg_qty:.1f} vs {overall_avg_qty:.1f} overall ({qty_diff:+.1f}% MORE)")
print(f"  Rating: {top_20_avg_rating:.2f} vs {overall_avg_rating:.2f} overall ({rating_diff:+.1f}%)")

# 6. Golden combination
print(f"\n⭐ Most Common 'Golden Combination' in Top 20%:")
combinations = {}
for i in np.where(top_20_mask)[0]:
    combo = (data[i, 0], data[i, 3], data[i, 1], data[i, 2])
    combinations[combo] = combinations.get(combo, 0) + 1

golden_combo = max(combinations.items(), key=lambda x: x[1])
print(f"  Branch: {golden_combo[0][0]}")
print(f"  Product: {golden_combo[0][1]}")
print(f"  Customer Type: {golden_combo[0][2]}")
print(f"  Gender: {golden_combo[0][3]}")
print(f"  Occurrences: {golden_combo[1]} times")

# 7. Percentile analysis
print(f"\n📊 Revenue Distribution by Percentile Groups:")
percentiles = [
    ("Top 20%", 0, 0.2),
    ("20-40%", 0.2, 0.4),
    ("40-60%", 0.4, 0.6),
    ("60-80%", 0.6, 0.8),
    ("Bottom 20%", 0.8, 1.0)
]

for name, start, end in percentiles:
    start_idx = int(len(sorted_indices) * start)
    end_idx = int(len(sorted_indices) * end)
    percentile_indices = sorted_indices[start_idx:end_idx]
    
    percentile_revenue = np.sum(total[percentile_indices])
    percentile_pct = (percentile_revenue / total_revenue) * 100
    avg_transaction = np.mean(total[percentile_indices])
    
    print(f"  {name:12s}: ${percentile_revenue:10.2f} ({percentile_pct:5.1f}% of total, "
          f"Avg: ${avg_transaction:.2f})")